In [1]:
import nibabel as nib
import numpy as numpy
import vtk
from vtk.util import numpy_support
import tomllib


with open("../src/config.toml", mode="rb") as f:
    config = tomllib.load(f)

base            = config["data"]["base"]
input_file      = base + config["data"]["nii_file"]
output_folder   = config["data"]["output_folder"]
output_file     = input_file[:-4] + ".vti"

In [2]:
def load_nifti(input_path, output_path, zoom_factor):
    """
    Crop a 3D NIfTI image and save it with updated affine.

    Parameters:
        input_path (str): input .nii/.nii.gz
        output_path (str): output file
        x_range, y_range, z_range: tuples (start, end)
    """

    nii = nib.load(input_path)
    data = numpy.array(nii.dataobj)  # efficient loading
    affine = nii.affine.copy()
    header = nii.header.copy()
    offset = affine[:3, 3] 

    pixel_spacing = header.get_zooms()[:3] 
    spacing_file = input_path[:-4] + "_pixel_spacing.txt"

    data_offset_file = input_path[:-4] + "_offset.txt"
    
    with open(spacing_file, 'w') as f:
        f.write(f"{pixel_spacing[0]} {pixel_spacing[1]} {pixel_spacing[2]}\n")
    print(f"Original pixel spacing saved to: {spacing_file} -> {pixel_spacing}")

    with open(data_offset_file, 'w') as f:
        f.write(f"{offset[0]} {offset[1]} {offset[2]}\n")
    print(f"Original pixel spacing saved to: {data_offset_file} -> {affine}")


    print("Original matrix shape = ", data.shape)

    return data, pixel_spacing, offset

data, pixel_spacing, offset = load_nifti(input_file, output_file, 1)
print("pixel_spacing = ", pixel_spacing)

Original pixel spacing saved to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/1601_pixel_spacing.txt -> (np.float32(0.75), np.float32(0.75), np.float32(0.7500004))
Original pixel spacing saved to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/1601_offset.txt -> [[-7.49396801e-01 -9.86332912e-03 -2.84118503e-02  1.23908943e+02]
 [ 1.08676692e-02 -7.49453902e-01 -2.64708530e-02  1.30018860e+02]
 [-2.80430261e-02 -2.68612392e-02  7.48994470e-01  1.59714651e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Original matrix shape =  (320, 320, 100)
pixel_spacing =  (np.float32(0.75), np.float32(0.75), np.float32(0.7500004))


/var/folders/l0/d5k4byxx3690l9b12qr7mm440000gp/T/ipykernel_7115/2957611650.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  data = numpy.array(nii.dataobj)  # efficient loading


In [ ]:
def write_vti(data, pixel_spacing, offset, output_file):
    vtk_image = vtk.vtkImageData()
    vtk_image.SetDimensions(data.shape)
    vtk_image.SetSpacing(pixel_spacing)
    vtk_image.SetOrigin(offset[0], offset[1], offset[2])

    # Flatten in Fortran order to match VTK's x-fastest convention
    flat_data = data.flatten(order="F")
    vtk_array = numpy_support.numpy_to_vtk(flat_data, deep=True)
    vtk_array.SetName("intensity")
    vtk_image.GetPointData().SetScalars(vtk_array)

    writer = vtk.vtkXMLImageDataWriter()
    writer.SetFileName(output_file)
    writer.SetInputData(vtk_image)
    writer.Write()
    print(f"Saved VTI file to: {output_file}")

# write_vti(data, pixel_spacing, -mesh_offset, output_file)
write_vti(data, pixel_spacing, offset*pixel_spacing, output_file)

Saved VTI file to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/1601.vti


## Segmentation mask

In [4]:
# import nrrd

# segment_folder = "segment/"
# filename = "Segmentation.seg.nrrd"

# eyeball_id = 3
# background_id = -1


# mask, header = nrrd.read(base + segment_folder + filename)
# origin = numpy.array(header["space origin"])
# print("origin = ", origin)

# print(mask.shape)

# write_vti(mask, pixel_spacing, -mesh_offset, base + segment_folder + filename[:-4] + ".vti" )